# Eka — `gita` persona QLoRA

EKA — GITA persona QLoRA fine-tune  |  Kaggle T4 (single GPU)
FULLY SELF-CONTAINED. Imports nothing from the Eka project. Upload this notebook to Kaggle and run top to bottom.
**BEFORE YOU RUN**
1. Kaggle -> Settings -> Accelerator = GPU T4 x1 (7B in 4-bit fits comfortably)
2. Kaggle -> Settings -> Internet = ON
3. Kaggle -> Add-ons -> Secrets, add all three:
       HF_TOKEN        (write permission)
       WANDB_API_KEY
       HF_USERNAME
4. No license to accept. Qwen/Qwen2.5-7B-Instruct is ungated and starts
   downloading immediately. (The base model was
   meta-llama/Meta-Llama-3-8B-Instruct until 2026-08-13; Meta approval for
   Llama 3.1/3.2 was still pending, so this moved to Qwen.)
5. Kaggle -> Save Version -> "Save & Run All (Commit)" so it keeps training
   after you close the browser. Kaggle sessions cap at 12h; this run needs
   far less, but the checkpoint-resume logic below survives a restart anyway.
ESTIMATED TIME ON A SINGLE T4
    ~565 train examples, effective batch 16, 3 epochs  =  ~105 optimizer steps
    ~55-70 s per optimizer step at seq len 2048        =  ~1.5-2 hrs
    plus ~15 min model download on first run
If you raise MAX_SEQ_LEN or EPOCHS, scale that estimate linearly.
**TO TRAIN A DIFFERENT PERSONA**
This file IS train_founder_lora_kaggle.py with MODE pre-set. Every persona
gets its own file on purpose: self-contained notebooks are safer on Kaggle
than one file you have to remember to edit before each run.

---

**This notebook is generated.** Edit `training/train_gita_lora_kaggle.py` and re-run `python scripts/build_kaggle_notebooks.py`; edits made here are overwritten on the next build.

## Setup — install pinned dependencies

Restart-safe: re-running this cell is a no-op once the versions match.

In [ ]:
%%capture
!pip install -q transformers==4.41.0 \
    peft==0.11.1 \
    trl==0.8.6 \
    datasets==2.19.1 \
    accelerate==0.30.0 \
    bitsandbytes==0.43.1 \
    wandb==0.17.0 \
    huggingface-hub==0.23.2

import os
# SECTION 1 below re-installs unless this is set; the cell above already did it.
os.environ["EKA_SKIP_INSTALL"] = "1"

## Section 1 — INSTALL

In a notebook, put this in the first cell prefixed with %%capture

In [ ]:
# %%capture
# !pip install -q transformers==4.41.0 peft==0.11.1 trl==0.8.6 \
#     datasets==2.19.1 accelerate==0.30.0 bitsandbytes==0.43.1 \
#     wandb==0.17.0 huggingface-hub==0.23.2

import os
import subprocess
import sys


def _pip_install() -> None:
    """Idempotent install so the script works as a plain .py run too."""
    packages = [
        "transformers==4.41.0",
        "peft==0.11.1",
        "trl==0.8.6",
        "datasets==2.19.1",
        "accelerate==0.30.0",
        "bitsandbytes==0.43.1",
        "wandb==0.17.0",
        "huggingface-hub==0.23.2",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages], check=False
    )


if os.environ.get("EKA_SKIP_INSTALL") != "1":
    _pip_install()

## Section 2 — AUTH (Kaggle Secrets, with plain-env fallback)

In [ ]:
def _load_secrets() -> dict:
    """Read HF_TOKEN / WANDB_API_KEY / HF_USERNAME from Kaggle Secrets or env."""
    names = ["HF_TOKEN", "WANDB_API_KEY", "HF_USERNAME"]
    found = {}
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for name in names:
            try:
                found[name] = client.get_secret(name)
            except Exception:
                found[name] = os.environ.get(name, "")
    except Exception:
        for name in names:
            found[name] = os.environ.get(name, "")

    for name, value in found.items():
        if value:
            os.environ[name] = value

    missing = [n for n in ("HF_TOKEN", "HF_USERNAME") if not found.get(n)]
    if missing:
        raise SystemExit(
            f"Missing required secret(s): {', '.join(missing)}\n"
            "Add them under Kaggle -> Add-ons -> Secrets, then restart the session."
        )
    return found


SECRETS = _load_secrets()

from huggingface_hub import HfApi, login  # noqa: E402

login(token=SECRETS["HF_TOKEN"])
print("✓ Hugging Face authenticated")

USE_WANDB = bool(SECRETS.get("WANDB_API_KEY"))
if USE_WANDB:
    import wandb

    wandb.login(key=SECRETS["WANDB_API_KEY"])
    print("✓ WandB authenticated")
else:
    print("! WANDB_API_KEY not set — training without experiment tracking")

## Section 3 — CONFIG

In [ ]:
MODE = "gita"  # <-- the ONLY line that differs from train_founder_lora_kaggle.py

HF_USERNAME = os.environ["HF_USERNAME"]
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATASET_REPO = f"{HF_USERNAME}/eka-datasets"
OUTPUT_REPO = f"{HF_USERNAME}/eka-{MODE}-qwen"
OUTPUT_DIR = f"/kaggle/working/{MODE}_lora"

MAX_SEQ_LEN = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8  # effective batch = 16
LR = 2e-4
SAVE_STEPS = 50
EVAL_STEPS = 50
WANDB_PROJECT = "eka"
RUN_NAME = f"eka-{MODE}-qwen-v1"

os.makedirs(OUTPUT_DIR, exist_ok=True)
if USE_WANDB:
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT

print(f"\n{'=' * 70}")
print(f"  EKA {MODE.upper()} LoRA")
print(f"  base    : {BASE_MODEL}")
print(f"  data    : {DATASET_REPO}  ({MODE}_train.jsonl / {MODE}_val.jsonl)")
print(f"  output  : {OUTPUT_REPO}")
print(f"{'=' * 70}\n")

import torch  # noqa: E402

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Kaggle -> Settings -> Accelerator -> GPU T4 x1.\n"
        "(4-bit QLoRA on a 7B model is not viable on CPU.)"
    )

GPU_NAME = torch.cuda.get_device_name(0)
# Turing (T4) has no bfloat16 support. Ampere+ (A100/L4) does. Picking the
# wrong one here is the most common cause of "RuntimeError: expected scalar
# type" or silent NaN losses on Kaggle.
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
print(f"GPU: {GPU_NAME}")
print(f"     bf16 supported: {SUPPORTS_BF16} -> compute dtype {COMPUTE_DTYPE}")
print(f"     VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

## Section 4 — CHECKPOINT RESUME

Kaggle sessions die. This makes a restart cost minutes, not hours.

In [ ]:
import glob  # noqa: E402


def find_latest_checkpoint(directory: str):
    checkpoints = glob.glob(os.path.join(directory, "checkpoint-*"))
    checkpoints = [c for c in checkpoints if os.path.isdir(c)]
    if not checkpoints:
        return None

    def step_of(path: str) -> int:
        try:
            return int(os.path.basename(path).split("-")[-1])
        except ValueError:
            return -1

    return max(checkpoints, key=step_of)


RESUME_FROM = find_latest_checkpoint(OUTPUT_DIR)
if RESUME_FROM:
    print(f"↻ Resuming from {RESUME_FROM}")
else:
    print("→ Starting fresh training")

## Section 5 — LOAD DATA

In [ ]:
from datasets import load_dataset  # noqa: E402

dataset = load_dataset(
    DATASET_REPO,
    data_files={
        "train": f"{MODE}_train.jsonl",
        "validation": f"{MODE}_val.jsonl",
    },
    token=os.environ["HF_TOKEN"],
)
print(f"\nTrain: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

if "text" not in dataset["train"].column_names:
    raise SystemExit(
        f"Expected a 'text' column, got {dataset['train'].column_names}.\n"
        "Re-run ml/scripts/preprocess.py then ml/scripts/upload_to_hf.py."
    )

print("\n--- one training example (truncated) ---")
print(dataset["train"][0]["text"][:600])
print("---------------------------------------\n")

## Section 6 — LOAD MODEL (4-bit NF4 quantization)

In [ ]:
from transformers import (  # noqa: E402
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=os.environ["HF_TOKEN"])

# Do NOT set pad_token = eos_token here. On Llama-3 that was harmless, because
# its eos (<|end_of_text|>) is a different token from the one the template ends
# turns with (<|eot_id|>). On Qwen2.5 they are the same token: eos IS <|im_end|>.
# The SFT collator masks pad positions out of the labels, so pad == eos would
# mask every stop token the model is supposed to be learning, and the adapter
# would never stop generating. Qwen ships a separate <|endoftext|> for padding.
if tokenizer.pad_token is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
    if "<|endoftext|>" in tokenizer.get_vocab():
        tokenizer.pad_token = "<|endoftext|>"
    else:
        print("! no distinct pad token found — stop tokens may be masked in labels")
print(f"     pad={tokenizer.pad_token!r} ({tokenizer.pad_token_id})  "
      f"eos={tokenizer.eos_token!r} ({tokenizer.eos_token_id})")
tokenizer.padding_side = "right"  # left padding corrupts causal LM training

print("Loading base model in 4-bit (first run downloads ~15GB)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},  # pin to GPU 0; "auto" can spill layers to CPU on T4
    trust_remote_code=True,
    token=os.environ["HF_TOKEN"],
)
model.config.use_cache = False  # incompatible with gradient checkpointing
model.config.pretraining_tp = 1
print("✓ Base model loaded")

## Section 7 — LORA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # noqa: E402

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    # All attention + MLP projections. Attention-only (q,v) trains faster but
    # learns persona style noticeably worse.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # expect ~40M trainable / ~0.5% of 7B

## Section 8 — TRAIN

In [ ]:
from trl import SFTTrainer  # noqa: E402

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    weight_decay=0.001,
    # Match the quantization compute dtype or loss goes NaN on T4.
    fp16=not SUPPORTS_BF16,
    bf16=SUPPORTS_BF16,
    optim="paged_adamw_8bit",
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,  # Kaggle /kaggle/working is capped at 20GB
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb" if USE_WANDB else "none",
    run_name=RUN_NAME,
    seed=42,
    group_by_length=True,  # big speedup: batches similar-length sequences
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,  # persona examples are short; packing blurs turn boundaries
)

print(f"\n{'=' * 70}")
print(f"  TRAINING START — {RUN_NAME}")
print(f"  ~{len(dataset['train']) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)} optimizer steps")
print(f"{'=' * 70}\n")

trainer.train(resume_from_checkpoint=RESUME_FROM)

metrics = trainer.evaluate()
print(f"\n✓ Final eval loss: {metrics.get('eval_loss'):.4f}")

## Section 9 — SAVE + PUSH TO HUB

In [ ]:
print("\nSaving adapter...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

card = f"""---
base_model: {BASE_MODEL}
library_name: peft
tags:
- eka
- lora
- qlora
- {MODE}
---

# eka-{MODE}-qwen

QLoRA adapter giving Qwen2.5-7B-Instruct Eka's **{MODE}** persona.

| | |
|---|---|
| base | `{BASE_MODEL}` |
| rank / alpha | {LORA_R} / {LORA_ALPHA} |
| epochs | {EPOCHS} |
| effective batch | {BATCH_SIZE * GRAD_ACCUM} |
| lr / schedule | {LR} cosine |
| max seq len | {MAX_SEQ_LEN} |
| train / val | {len(dataset['train'])} / {len(dataset['validation'])} |
| final eval loss | {metrics.get('eval_loss', float('nan')):.4f} |
| trained on | {GPU_NAME} |

Merge and serve with `ml/scripts/merge_lora.py --mode {MODE}`.
"""
with open(os.path.join(OUTPUT_DIR, "README.md"), "w", encoding="utf-8") as handle:
    handle.write(card)

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=OUTPUT_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=OUTPUT_REPO,
    # Checkpoints are large and already superseded by the final adapter.
    ignore_patterns=["checkpoint-*", "*.pt", "runs/*"],
)
print(f"\n✅ Pushed to HF Hub: https://huggingface.co/{OUTPUT_REPO}")

## Section 10 — SANITY GENERATION

Does it actually sound like the persona? Read the output, don't trust the loss.

In [ ]:
PROBES = {
    "founder": "I have 3 paying customers at 2000/month and 4 months of runway. Should I raise?",
    "chanakya": "My business partner is hiding revenue numbers from me.",
    "gita": "I did everything right and still lost. What was the point?",
    "reflection": "I keep quitting things right before they start working.",
}

model.eval()
prompt = (
    "<|im_start|>user\n"
    f"{PROBES[MODE]}<|im_end|>\n"
    "<|im_start|>assistant\n"
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        top_k=40,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

print(f"\n{'=' * 70}")
print(f"  SANITY CHECK — {MODE}")
print(f"{'=' * 70}")
print(f"USER: {PROBES[MODE]}\n")
print("EKA :", tokenizer.decode(output[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True))
print(f"{'=' * 70}\n")

if USE_WANDB:
    wandb.finish()

print(f"✅ {MODE} DONE. Next: train the remaining personas, then")
print(f"   python ml/scripts/merge_lora.py --mode {MODE}")

---

## Done

Confirm the adapter actually landed on the Hub before you count this run as finished — a repo with only a `README.md` means the push failed:

```bash
python -c "
from huggingface_hub import HfApi; import os
from dotenv import load_dotenv; load_dotenv('.env')
print(HfApi(token=os.getenv('HF_TOKEN')).list_repo_files(
    'amijackofalltrades/eka-gita-qwen'))"
```

Then merge locally: `python ml/scripts/merge_lora.py --mode gita`